<a href="https://colab.research.google.com/github/sparshbansal-newton/deep-learning-labs/blob/main/Notebooks/4_pytorch_forward_prop/churn_forward_prop_and_loss_SOLUTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Network with PyTorch — Forward Propagation & Loss (Bank Customer Churn)

In this notebook we build the **first part** of a neural network from scratch with PyTorch, using the **Bank Customer Churn** dataset. Each row is a bank customer; our job is to predict whether the customer **left the bank** (`Churn = 1`) or **stayed** (`Churn = 0`).

We focus on only **two building blocks** today:
1. **Forward propagation** — how the model turns input features into a predicted probability.
2. **Loss** — how we measure how wrong that prediction is.

We will **not** train the model yet (no backpropagation, no gradient updates). The weights stay **random** throughout — the goal is purely to understand *what a forward pass computes* and *what the loss number means*. Training comes next class.

Before any of that, real-world data needs **cleaning**. This dataset forces us to practise three essential steps:
- **Removing unnecessary columns** (IDs and names carry no predictive signal)
- **Label encoding** (a neural network only understands numbers, not text like `France` or `Male`)
- **Standardization** (features live on wildly different scales — Age ~40 vs Balance ~100,000)

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Fix the random seed so weight initialization is reproducible across runs
torch.manual_seed(42)

### Load the data

We read the CSV straight from a public URL into a pandas DataFrame and peek at the first few rows.

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv')
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,Num Of Products,Has Credit Card,Is Active Member,Estimated Salary,Churn
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
df.shape

(10000, 13)

### Step 1 — Remove unnecessary columns

Look at the columns above. `CustomerId` and `Surname` **identify** a customer but say nothing about whether they will churn — feeding them to the model would only add noise (and let it 'memorise' individuals). We drop them so only genuinely predictive features remain.

> **Hint:** use `df.drop(columns=[...], inplace=True)`.

In [4]:
df.drop(columns=['CustomerId', 'Surname'], inplace=True)

In [5]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,Num Of Products,Has Credit Card,Is Active Member,Estimated Salary,Churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Step 2 — Label encoding (text → numbers)

Two columns are **text**: `Geography` (France / Spain / Germany) and `Gender` (Female / Male). A neural network multiplies inputs by weights, so every feature must be **numeric**. `LabelEncoder` maps each category to an integer (e.g. Female→0, Male→1).

> **Hint:** create a `LabelEncoder()` and call `.fit_transform()` on each text column, assigning the result back to that column.

*(Note: our target `Churn` is already 0/1, so it needs no encoding.)*

In [6]:
df['Geography'] = LabelEncoder().fit_transform(df['Geography'])
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])

In [7]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,Num Of Products,Has Credit Card,Is Active Member,Estimated Salary,Churn
0,619,0,0,42,2,0.00,1,1,1,101348.88,1
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,0,39,1,0.00,2,0,0,93826.63,0
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0


### Step 3 — Train / test split

We separate the **features** (everything except `Churn`) from the **target** (`Churn`), then hold out 20% of the rows as a test set. `random_state=42` makes the split reproducible.

> **Hint:** features `X = df.drop(columns=['Churn'])`, target `y = df['Churn']`, then `train_test_split(X, y, test_size=0.2, random_state=42)`.

In [8]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Step 4 — Standardization (scaling)

Features are on very different scales (`Age` ~40, `Balance` ~100000, `Credit Score` ~600). Large-scale features would dominate the weighted sum and slow learning. `StandardScaler` rescales each feature to **mean 0, standard deviation 1**.

> **Important:** `fit_transform` on the **training** set only, then `transform` the test set with the *same* scaler — the test set must never influence the scaling statistics.

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
X_train

array([[ 0.35649971, -0.9055496 ,  0.91324755, ...,  0.64920267,
         0.97481699,  1.36766974],
       [-0.20389777,  0.30164867,  0.91324755, ...,  0.64920267,
         0.97481699,  1.6612541 ],
       [-0.96147213,  1.50884694,  0.91324755, ...,  0.64920267,
        -1.02583358, -0.25280688],
       ...,
       [ 0.86500853, -0.9055496 , -1.09499335, ..., -1.54035103,
        -1.02583358, -0.1427649 ],
       [ 0.15932282, -0.9055496 ,  0.91324755, ...,  0.64920267,
        -1.02583358, -0.05082558],
       [ 0.47065475,  0.30164867,  0.91324755, ...,  0.64920267,
         0.97481699, -0.81456811]], shape=(8000, 10))

### Step 5 — NumPy arrays → PyTorch tensors

PyTorch operates on **tensors**, so we convert our NumPy arrays. Note that `y_train` / `y_test` are pandas Series, so we take their `.values` first.

> **Hint:** `torch.from_numpy(...)`. For the labels use `y_train.values` and `y_test.values`.

In [11]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train.values)
y_test_tensor = torch.from_numpy(y_test.values)

In [12]:
X_train_tensor.shape

torch.Size([8000, 10])

In [13]:
y_train_tensor.shape

torch.Size([8000])

### Step 6 — Defining the model

A single-layer neural network does two things in its `forward` pass:

1. **Linear step** — combine all input features into one number:
$$z = X \cdot \text{weights} + \text{bias}$$
2. **Activation step** — squash that number into a probability between 0 and 1 with the sigmoid function:
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

This `forward` pass alone doesn't learn — it just computes a prediction from whatever weights it currently has (right now: random values).

The `loss_function` then compares the prediction $\hat{y}$ to the true label $y$ using **binary cross-entropy**:
$$L = -\big(y \cdot \log(\hat{y}) + (1-y)\cdot \log(1-\hat{y})\big)$$

A high loss means the prediction was far from the truth; a low loss means it was close. We do **not** set `requires_grad` here — no gradients or updates happen this class.

> **Hint:** in `__init__`, `self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64)` and `self.bias = torch.zeros(1, dtype=torch.float64)`. In `forward`, use `torch.matmul` then `torch.sigmoid`.

In [14]:
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64)
    self.bias = torch.zeros(1, dtype=torch.float64)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss (binary cross-entropy)
    loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
    return loss

### Step 7 — Running a single forward pass

We create the model (random weights), run **one** forward pass on the training data, and compute the loss. No epochs, no weight updates — just to see what a single forward pass + loss looks like.

> **Hint:** `model = MySimpleNN(X_train_tensor)`, then `model.forward(...)`, then `model.loss_function(...)`.

In [15]:
# create model
model = MySimpleNN(X_train_tensor)

# forward pass (single pass, no training loop)
y_pred_train = model.forward(X_train_tensor)

# loss calculation
loss = model.loss_function(y_pred_train, y_train_tensor)

print(f'Predicted probabilities (first 5): {y_pred_train[:5].squeeze().tolist()}')
print(f'Loss with random (untrained) weights: {loss.item()}')

Predicted probabilities (first 5): [0.7826420199658184, 0.8766861593336313, 0.40480540262348463, 0.6549078020991622, 0.2246916440604492]
Loss with random (untrained) weights: 0.9097758182792899


### Evaluation

Let's check accuracy on the test set. The weights were never updated, so they are still random — expect accuracy around chance level. This is the motivation for the next class: we need a way to **update** the weights so the loss goes down and accuracy goes up — that's what backpropagation and gradient descent will do.

In [16]:
# model evaluation (weights are still random/untrained at this point)
with torch.no_grad():
  y_pred_test = model.forward(X_test_tensor)
  y_pred_labels = (y_pred_test > 0.5).float()
  accuracy = (y_pred_labels.squeeze() == y_test_tensor).float().mean()
  print(f'Accuracy with untrained (random) weights: {accuracy.item()}')

Accuracy with untrained (random) weights: 0.46650001406669617


### What's next

In the next class we'll add:
- `requires_grad=True` on the weights and bias
- A training loop that calls `loss.backward()` to compute gradients (backpropagation)
- Gradient-descent updates to actually reduce the loss over multiple epochs

By comparing the loss/accuracy here (random weights) to the loss/accuracy after training, you'll see directly what training accomplishes.